In [ ]:
!pip install open3d datasets plyfile huggingface_hub plotly wandb torch-geometric tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HUGGINGFACE')

from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="/content/MaizeField3D"
)

import zipfile

zip_path = "/content/MaizeField3D/datasets/FielGrwon_ZeaMays_RawPCD_10k.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("./MaizeField3D/extracted")

print("Unzipped!")

import os

data_path = "/content/MaizeField3D/extracted"
ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print(f"Total PLY files: {len(ply_files)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Unzipped!
Total PLY files: 1045


Normalize point cloud

In [ ]:
import os
import torch
import numpy as np
import open3d as o3d
from torch.utils.data import Dataset

NUM_POINTS = 2048

class MaizeDataset(Dataset):
    def __init__(self, ply_files, augment=True):
        self.files = ply_files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        pcd = o3d.io.read_point_cloud(self.files[idx])
        points = np.asarray(pcd.points)

        # Sampling
        if len(points) >= NUM_POINTS:
            idxs = np.random.choice(len(points), NUM_POINTS, replace=False)
        else:
            idxs = np.random.choice(len(points), NUM_POINTS, replace=True)
        points = points[idxs]

        # Normalize (VERY IMPORTANT)
        centroid = points.mean(axis=0)
        points = points - centroid
        scale = np.max(np.linalg.norm(points, axis=1))
        points = points / (scale + 1e-6)

        #  Augmentation (improves generalization)
        if self.augment:
            points += 0.01 * np.random.randn(*points.shape)

        return torch.tensor(points, dtype=torch.float32)

Transformer Autoencoder

In [ ]:
import torch.nn as nn

class PointTransformerAE(nn.Module):
    def __init__(self, emb_dim=128, num_heads=4, depth=4):
        super().__init__()

        self.input_proj = nn.Linear(3, emb_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True,
            norm_first=True   #  stability improvement
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.pool = nn.AdaptiveMaxPool1d(1)

        #  Improved decoder
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.ReLU(),

            nn.Linear(512, NUM_POINTS * 3),
            nn.Tanh()   #  keeps outputs bounded
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)

        x = x.transpose(1, 2)
        x = self.pool(x).squeeze(-1)

        x = self.decoder(x)
        x = x.view(-1, NUM_POINTS, 3)

        return x

Chamfer Distance Loss

In [ ]:
import torch

def chamfer_distance(p1, p2):
    dist = torch.cdist(p1, p2)
    return dist.min(dim=2)[0].mean() + dist.min(dim=1)[0].mean()

In [ ]:
import wandb
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

# -----------------------
# W&B INIT
# -----------------------
wandb.init(project="maize-transformer-ae", config={
    "epochs": 100,
    "batch_size": 8,
    "lr": 1e-3,
    "points": 2048
})

# -----------------------
# DATA
# -----------------------
dataset = MaizeDataset(ply_files, augment=True)
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

# -----------------------
# MODEL
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PointTransformerAE().to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100
)
#  LR warmup
def get_lr(epoch):
    if epoch < 5:
        return 1e-4
    return 1e-3

/tmp/ipykernel_952/1939747806.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)


In [ ]:
EPOCHS = 100

# -----------------------
# TRAIN
# -----------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # Apply warmup LR
    for param_group in optimizer.param_groups:
        param_group["lr"] = get_lr(epoch)

    for points in loader:
        points = points.to(device)

        recon = model(points)
        loss = chamfer_distance(points, recon)

        optimizer.zero_grad()
        loss.backward()

        #  Gradient clipping (CRITICAL)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(loader)

    wandb.log({
        "epoch": epoch,
        "loss": avg_loss,
        "lr": optimizer.param_groups[0]["lr"]
    })

    print(f"Epoch {epoch}: Loss = {avg_loss:.4f}")

Epoch 0: Loss = 0.1267
Epoch 1: Loss = 0.1048
Epoch 2: Loss = 0.1036
Epoch 3: Loss = 0.1024
Epoch 4: Loss = 0.1011
Epoch 5: Loss = 0.1072
Epoch 6: Loss = 0.0901
Epoch 7: Loss = 0.0842
Epoch 8: Loss = 0.0829
Epoch 9: Loss = 0.0810
Epoch 10: Loss = 0.0794
Epoch 11: Loss = 0.0787
Epoch 12: Loss = 0.0767
Epoch 13: Loss = 0.0761
Epoch 14: Loss = 0.0744
Epoch 15: Loss = 0.0743
Epoch 16: Loss = 0.0733
Epoch 17: Loss = 0.0728
Epoch 18: Loss = 0.0721
Epoch 19: Loss = 0.0709
Epoch 20: Loss = 0.0715
Epoch 21: Loss = 0.0700
Epoch 22: Loss = 0.0701
Epoch 23: Loss = 0.0692
Epoch 24: Loss = 0.0683
Epoch 25: Loss = 0.0679
Epoch 26: Loss = 0.0669
Epoch 27: Loss = 0.0677
Epoch 28: Loss = 0.0669
Epoch 29: Loss = 0.0661
Epoch 30: Loss = 0.0656
Epoch 31: Loss = 0.0653
Epoch 32: Loss = 0.0657
Epoch 33: Loss = 0.0649
Epoch 34: Loss = 0.0646
Epoch 35: Loss = 0.0643
Epoch 36: Loss = 0.0637
Epoch 37: Loss = 0.0636
Epoch 38: Loss = 0.0634
Epoch 39: Loss = 0.0627
Epoch 40: Loss = 0.0624
Epoch 41: Loss = 0.0621
Ep

Visualization ( Original vs Reconstructed )

In [ ]:
import plotly.graph_objects as go
import torch

def visualize_original_vs_recon(model, loader, device):
    model.eval()

    with torch.no_grad():
        points = next(iter(loader))  # take one batch
        points = points.to(device)

        recon = model(points)

        # Take first sample
        orig = points[5].cpu().numpy()
        rec = recon[5].cpu().numpy()

    # Shift reconstruction to right (so no overlap)
    shift = 2.5  # increase if still overlapping
    rec[:, 0] += shift

    fig = go.Figure()

    # Original
    fig.add_trace(go.Scatter3d(
        x=orig[:, 0],
        y=orig[:, 1],
        z=orig[:, 2],
        mode='markers',
        marker=dict(size=2),
        name='Original'
    ))

    # Reconstructed
    fig.add_trace(go.Scatter3d(
        x=rec[:, 0],
        y=rec[:, 1],
        z=rec[:, 2],
        mode='markers',
        marker=dict(size=2),
        name='Reconstructed'
    ))

    fig.update_layout(
        title="Original vs Reconstructed (Side by Side)",
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        showlegend=True
    )

    fig.show()

In [ ]:
visualize_original_vs_recon(model, loader, device)

Save Model

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': epoch
}, 'transformer_ae_maize.pth')

Load Model

In [ ]:
model = PointTransformerAE().to(device)

checkpoint = torch.load('transformer_ae_maize.pth', map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

/tmp/ipykernel_952/1939747806.py:18: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True



PointTransformerAE(
  (input_proj): Linear(in_features=3, out_features=128, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (pool): AdaptiveMaxPool1d(output_size=1)
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()